# Comparación grupal de agentes Connect-4

Este cuaderno compara los agentes ubicados en `tournament/groups` usando rutas relativas.

Cumple con el criterio solicitado:

1. Compara varios agentes: `Group A policy`, `Group B policy`, opcionalmente `Group C policy`, `Random`, `Center`, `rita_v1.py` y `rita_v2.py` si existen.
2. Evalúa múltiples combinaciones mediante round-robin.
3. Incluye una variable numérica de recursos: `num_iterations`.
4. Reporta win rate promedio, peor caso, tiempo promedio y robustez.
5. Genera conclusiones automáticas sobre superioridad relativa y potencial de los agentes del grupo.


## 1. Setup, rutas relativas y carga de políticas


In [ ]:
from pathlib import Path
import sys
import time
import math
import inspect
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PRIMARY_COLOR = "#234af3"

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


CURRENT_DIR = Path.cwd().resolve()


def find_groups_dir(start_dir: Path) -> Path:
    if start_dir.name == "Group A" and start_dir.parent.name == "groups":
        return start_dir.parent

    if start_dir.name == "groups":
        return start_dir

    candidate = start_dir / "tournament" / "groups"
    if candidate.exists():
        return candidate

    for parent in [start_dir] + list(start_dir.parents):
        candidate = parent / "tournament" / "groups"
        if candidate.exists():
            return candidate
        if parent.name == "groups":
            return parent

    raise FileNotFoundError("No se encontró la carpeta tournament/groups usando rutas relativas.")


GROUPS_DIR = find_groups_dir(CURRENT_DIR)

GROUP_A_DIR = GROUPS_DIR / "Group A"
GROUP_B_DIR = GROUPS_DIR / "Group B"
GROUP_C_DIR = GROUPS_DIR / "Group C"

A_POLICY_PATH = GROUP_A_DIR / "policy.py"
B_POLICY_PATH = GROUP_B_DIR / "policy.py"
C_POLICY_PATH = GROUP_C_DIR / "policy.py"

A_V1_PATH = GROUP_A_DIR / "rita_v1.py"
A_V2_PATH = GROUP_A_DIR / "rita_v2.py"

PROJECT_ROOT = None
for parent in [GROUPS_DIR] + list(GROUPS_DIR.parents):
    if (parent / "tournament").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = GROUPS_DIR.parents[1]

TOURNAMENT_ROOT = PROJECT_ROOT / "tournament"

for path in [PROJECT_ROOT, TOURNAMENT_ROOT]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))


def load_module_from_path(path: Path, module_name: str):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"No se pudo cargar el módulo desde {path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    return module


def find_policy_class(module, preferred_names=None):
    if preferred_names is None:
        preferred_names = []

    for name in preferred_names:
        if hasattr(module, name):
            cls = getattr(module, name)
            if inspect.isclass(cls) and hasattr(cls, "mount") and hasattr(cls, "act"):
                return cls

    candidates = []
    for _, obj in inspect.getmembers(module, inspect.isclass):
        if obj.__module__ != module.__name__:
            continue
        if hasattr(obj, "mount") and hasattr(obj, "act"):
            candidates.append(obj)

    if not candidates:
        raise ValueError("No se encontró una clase compatible con mount() y act().")

    return sorted(candidates, key=lambda cls: cls.__name__)[0]


def load_policy_class(path: Path, label: str):
    safe_label = label.lower().replace(" ", "_").replace("-", "_").replace("/", "_")
    module = load_module_from_path(path, f"loaded_{safe_label}")

    cls = find_policy_class(
        module,
        preferred_names=[
            "Policy",
            "RitaVersion2",
            "RitaVersion1",
            "RitaV2",
            "RitaV1",
            "RitaAgentV1",
        ],
    )

    print(f"{label}: clase cargada -> {cls.__name__}")
    return cls


GroupAPolicyClass = load_policy_class(A_POLICY_PATH, "Group A policy")
GroupBPolicyClass = load_policy_class(B_POLICY_PATH, "Group B policy")

GroupCPolicyClass = None
if C_POLICY_PATH.exists():
    GroupCPolicyClass = load_policy_class(C_POLICY_PATH, "Group C policy")

RitaV1Class = None
if A_V1_PATH.exists():
    RitaV1Class = load_policy_class(A_V1_PATH, "Rita V1")

RitaV2Class = None
if A_V2_PATH.exists():
    RitaV2Class = load_policy_class(A_V2_PATH, "Rita V2")

print("CURRENT_DIR:", CURRENT_DIR)
print("GROUPS_DIR:", GROUPS_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("A_POLICY_PATH:", A_POLICY_PATH, A_POLICY_PATH.exists())
print("B_POLICY_PATH:", B_POLICY_PATH, B_POLICY_PATH.exists())
print("C_POLICY_PATH:", C_POLICY_PATH, C_POLICY_PATH.exists())
print("A_V1_PATH:", A_V1_PATH, A_V1_PATH.exists())
print("A_V2_PATH:", A_V2_PATH, A_V2_PATH.exists())


## 2. Motor local de Connect-4


In [ ]:
ROWS = 6
COLS = 7
EMPTY = 0
P1 = -1
P2 = 1


def new_board():
    return np.zeros((ROWS, COLS), dtype=int)


def legal_actions(board):
    return [col for col in range(COLS) if board[0, col] == EMPTY]


def current_player(board):
    p1_count = int(np.sum(board == P1))
    p2_count = int(np.sum(board == P2))
    return P1 if p1_count <= p2_count else P2


def opponent(player):
    return P2 if player == P1 else P1


def apply_move(board, col, player):
    if col not in legal_actions(board):
        raise ValueError(f"Acción ilegal: {col}. Legales: {legal_actions(board)}")

    new = board.copy()

    for row in range(ROWS - 1, -1, -1):
        if new[row, col] == EMPTY:
            new[row, col] = player
            return new

    raise ValueError(f"Columna llena: {col}")


def check_winner(board):
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

    for row in range(ROWS):
        for col in range(COLS):
            player = board[row, col]
            if player == EMPTY:
                continue

            for dr, dc in directions:
                count = 0
                for step in range(4):
                    r = row + dr * step
                    c = col + dc * step

                    if 0 <= r < ROWS and 0 <= c < COLS and board[r, c] == player:
                        count += 1
                    else:
                        break

                if count == 4:
                    return int(player)

    return 0


def is_terminal(board):
    return check_winner(board) != 0 or len(legal_actions(board)) == 0


def ci95(values):
    values = np.array(values, dtype=float)

    if len(values) <= 1:
        return 0.0

    return 1.96 * values.std(ddof=1) / np.sqrt(len(values))


## 3. Agentes auxiliares y wrappers


In [ ]:
class RandomPlayer:
    def __init__(self, seed=0):
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def mount(self, timeout=None):
        self.rng = np.random.default_rng(self.seed)

    def act(self, board):
        actions = legal_actions(board)
        if not actions:
            return 0
        return int(self.rng.choice(actions))


class CenterPlayer:
    def mount(self, timeout=None):
        pass

    def act(self, board):
        for col in [3, 2, 4, 1, 5, 0, 6]:
            if col in legal_actions(board):
                return int(col)

        return int(legal_actions(board)[0])


class ConfiguredPolicy:
    def __init__(self, policy_cls, name=None, seed=0, **params):
        self.policy_cls = policy_cls
        self.name = name or policy_cls.__name__
        self.seed = seed
        self.params = params
        self.inner = None

    def mount(self, timeout=None):
        self.inner = self.policy_cls()

        try:
            self.inner.mount(timeout)
        except TypeError:
            self.inner.mount()

        if hasattr(self.inner, "rng"):
            self.inner.rng = np.random.default_rng(self.seed)

        for key, value in self.params.items():
            setattr(self.inner, key, value)

    def act(self, board):
        if self.inner is None:
            self.mount()

        return int(self.inner.act(board))


def make_external_policy(policy_cls, name, seed=0, **params):
    return ConfiguredPolicy(
        policy_cls=policy_cls,
        name=name,
        seed=seed,
        **params,
    )


def mount_policy(policy, timeout=1.0):
    try:
        policy.mount(timeout)
    except TypeError:
        policy.mount()


## 4. Motor de evaluación


In [ ]:
def play_game(first_policy, second_policy, timeout=1.0, seed=0, max_turns=42):
    np.random.seed(seed)

    mount_policy(first_policy, timeout)
    mount_policy(second_policy, timeout)

    board = new_board()
    player = P1

    p1_time = 0.0
    p2_time = 0.0
    p1_actions = 0
    p2_actions = 0

    for turn in range(max_turns):
        policy = first_policy if player == P1 else second_policy
        actions = legal_actions(board)

        t0 = time.perf_counter()

        try:
            action = int(policy.act(board.copy()))
        except Exception as e:
            winner = opponent(player)
            return {
                "winner_piece": winner,
                "reason": f"error: {e}",
                "moves": turn,
                "p1_avg_time": p1_time / max(p1_actions, 1),
                "p2_avg_time": p2_time / max(p2_actions, 1),
            }

        elapsed = time.perf_counter() - t0

        if player == P1:
            p1_time += elapsed
            p1_actions += 1
        else:
            p2_time += elapsed
            p2_actions += 1

        if action not in actions:
            winner = opponent(player)
            return {
                "winner_piece": winner,
                "reason": f"illegal action {action}",
                "moves": turn,
                "p1_avg_time": p1_time / max(p1_actions, 1),
                "p2_avg_time": p2_time / max(p2_actions, 1),
            }

        board = apply_move(board, action, player)
        winner = check_winner(board)

        if winner != 0:
            return {
                "winner_piece": winner,
                "reason": "normal",
                "moves": turn + 1,
                "p1_avg_time": p1_time / max(p1_actions, 1),
                "p2_avg_time": p2_time / max(p2_actions, 1),
            }

        if len(legal_actions(board)) == 0:
            return {
                "winner_piece": 0,
                "reason": "draw",
                "moves": turn + 1,
                "p1_avg_time": p1_time / max(p1_actions, 1),
                "p2_avg_time": p2_time / max(p2_actions, 1),
            }

        player = opponent(player)

    return {
        "winner_piece": 0,
        "reason": "max_turns",
        "moves": max_turns,
        "p1_avg_time": p1_time / max(p1_actions, 1),
        "p2_avg_time": p2_time / max(p2_actions, 1),
    }


def evaluate_matchup(
    name_a,
    factory_a,
    name_b,
    factory_b,
    n_games=20,
    timeout=1.0,
    alternate_colors=True,
):
    rows = []

    for game in range(n_games):
        if alternate_colors and game % 2 == 1:
            first_name = name_b
            second_name = name_a
            first_policy = factory_b(game)
            second_policy = factory_a(game)
        else:
            first_name = name_a
            second_name = name_b
            first_policy = factory_a(game)
            second_policy = factory_b(game)

        result = play_game(
            first_policy=first_policy,
            second_policy=second_policy,
            timeout=timeout,
            seed=game,
        )

        if result["winner_piece"] == P1:
            winner = first_name
        elif result["winner_piece"] == P2:
            winner = second_name
        else:
            winner = "Draw"

        rows.append({
            "game": game + 1,
            "first_player": first_name,
            "second_player": second_name,
            "winner": winner,
            "winner_piece": result["winner_piece"],
            "reason": result["reason"],
            "moves": result["moves"],
            "first_avg_time": result["p1_avg_time"],
            "second_avg_time": result["p2_avg_time"],
        })

    return pd.DataFrame(rows)


def summarize_agent(df, agent_name):
    wins = int((df["winner"] == agent_name).sum())
    draws = int((df["winner"] == "Draw").sum())
    losses = int(((df["winner"] != agent_name) & (df["winner"] != "Draw")).sum())
    total = len(df)

    return {
        "agent": agent_name,
        "games": total,
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "win_rate": wins / total,
        "draw_rate": draws / total,
        "loss_rate": losses / total,
        "avg_moves": float(df["moves"].mean()),
    }


## 5. Suite de agentes


In [ ]:
AGENTS = []

AGENTS.append((
    "Group A policy",
    lambda seed: make_external_policy(
        GroupAPolicyClass,
        name="Group A policy",
        seed=seed,
    ),
))

AGENTS.append((
    "Group B policy",
    lambda seed: make_external_policy(
        GroupBPolicyClass,
        name="Group B policy",
        seed=seed,
    ),
))

if GroupCPolicyClass is not None:
    AGENTS.append((
        "Group C policy",
        lambda seed: make_external_policy(
            GroupCPolicyClass,
            name="Group C policy",
            seed=seed,
        ),
    ))

AGENTS.append(("Random", lambda seed: RandomPlayer(seed=seed)))
AGENTS.append(("Center", lambda seed: CenterPlayer()))

if RitaV1Class is not None:
    AGENTS.append((
        "Rita V1",
        lambda seed: make_external_policy(
            RitaV1Class,
            name="Rita V1",
            seed=seed,
            num_iterations=350,
            time_limit=0.7,
        ),
    ))

if RitaV2Class is not None:
    AGENTS.append((
        "Rita V2",
        lambda seed: make_external_policy(
            RitaV2Class,
            name="Rita V2",
            seed=seed,
            num_iterations=1000,
            time_limit=60 / 42,
        ),
    ))

print("Agentes incluidos:")
for name, _ in AGENTS:
    print("-", name)


## 6. Experimento 1: Round-robin entre agentes


In [ ]:
N_GAMES_ROUND_ROBIN = 12
TIMEOUT = 1.0

round_robin_rows = []

for i, (name_a, factory_a) in enumerate(AGENTS):
    for j, (name_b, factory_b) in enumerate(AGENTS):
        if i >= j:
            continue

        print(f"Evaluando: {name_a} vs {name_b}")

        df_match = evaluate_matchup(
            name_a,
            factory_a,
            name_b,
            factory_b,
            n_games=N_GAMES_ROUND_ROBIN,
            timeout=TIMEOUT,
            alternate_colors=True,
        )

        summary_a = summarize_agent(df_match, name_a)
        summary_b = summarize_agent(df_match, name_b)

        round_robin_rows.append({
            "agent": name_a,
            "opponent": name_b,
            "win_rate": summary_a["win_rate"],
            "draw_rate": summary_a["draw_rate"],
            "loss_rate": summary_a["loss_rate"],
            "avg_moves": summary_a["avg_moves"],
        })

        round_robin_rows.append({
            "agent": name_b,
            "opponent": name_a,
            "win_rate": summary_b["win_rate"],
            "draw_rate": summary_b["draw_rate"],
            "loss_rate": summary_b["loss_rate"],
            "avg_moves": summary_b["avg_moves"],
        })

df_round_robin = pd.DataFrame(round_robin_rows)

df_round_robin_summary = (
    df_round_robin
    .groupby("agent")
    .agg(
        mean_win_rate=("win_rate", "mean"),
        worst_win_rate=("win_rate", "min"),
        mean_loss_rate=("loss_rate", "mean"),
        mean_draw_rate=("draw_rate", "mean"),
        mean_moves=("avg_moves", "mean"),
    )
    .reset_index()
    .sort_values(["mean_win_rate", "worst_win_rate"], ascending=[False, False])
)

df_round_robin_summary


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    df_round_robin_summary["agent"],
    df_round_robin_summary["mean_win_rate"],
    color=PRIMARY_COLOR,
)

plt.ylim(0, 1.05)
plt.ylabel("Win rate promedio")
plt.title("Comparación grupal: desempeño promedio en round-robin")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
matrix = df_round_robin.pivot_table(
    index="agent",
    columns="opponent",
    values="win_rate",
    aggfunc="mean",
)

plt.figure(figsize=(10, 6))

plt.imshow(
    matrix,
    aspect="auto",
    origin="lower",
)

plt.colorbar(label="Win rate")

plt.xticks(
    ticks=np.arange(len(matrix.columns)),
    labels=matrix.columns,
    rotation=35,
    ha="right",
)

plt.yticks(
    ticks=np.arange(len(matrix.index)),
    labels=matrix.index,
)

plt.xlabel("Oponente")
plt.ylabel("Agente")
plt.title("Matriz de desempeño por combinación de agentes")
plt.show()

matrix


## 7. Experimento 2: Variable numérica de recursos


In [ ]:
RESOURCE_ITERATIONS = [50, 100, 250, 500, 800, 1000]
RESOURCE_TIME_LIMIT = 60 / 42

RESOURCE_OPPONENTS = [
    ("Random", lambda seed: RandomPlayer(seed=seed)),
    ("Center", lambda seed: CenterPlayer()),
    ("Group B policy", lambda seed: make_external_policy(
        GroupBPolicyClass,
        name="Group B policy",
        seed=seed,
    )),
]

if GroupCPolicyClass is not None:
    RESOURCE_OPPONENTS.append((
        "Group C policy",
        lambda seed: make_external_policy(
            GroupCPolicyClass,
            name="Group C policy",
            seed=seed,
        ),
    ))

N_REPEATS_RESOURCE = 10
GAMES_PER_REPEAT_RESOURCE = 6

resource_rows = []

for num_iterations in RESOURCE_ITERATIONS:
    config_name = f"Group A {num_iterations} it"

    print(f"\\nConfiguración: {config_name}")

    for opponent_name, opponent_factory in RESOURCE_OPPONENTS:
        print(f"  Contra {opponent_name}")

        repeat_win_rates = []
        repeat_loss_rates = []
        repeat_draw_rates = []
        repeat_times = []

        for repeat in range(N_REPEATS_RESOURCE):
            df = evaluate_matchup(
                config_name,
                lambda seed, n=num_iterations: make_external_policy(
                    GroupAPolicyClass,
                    name=config_name,
                    seed=seed + repeat * 100,
                    num_iterations=n,
                    time_limit=RESOURCE_TIME_LIMIT,
                ),
                opponent_name,
                opponent_factory,
                n_games=GAMES_PER_REPEAT_RESOURCE,
                timeout=TIMEOUT,
                alternate_colors=True,
            )

            summary = summarize_agent(df, config_name)

            time_values = []
            time_values.extend(
                df.loc[df["first_player"] == config_name, "first_avg_time"].tolist()
            )
            time_values.extend(
                df.loc[df["second_player"] == config_name, "second_avg_time"].tolist()
            )

            repeat_win_rates.append(summary["win_rate"])
            repeat_loss_rates.append(summary["loss_rate"])
            repeat_draw_rates.append(summary["draw_rate"])
            repeat_times.append(float(np.mean(time_values)) if time_values else np.nan)

        resource_rows.append({
            "config": config_name,
            "num_iterations": num_iterations,
            "opponent": opponent_name,
            "mean_win_rate": float(np.mean(repeat_win_rates)),
            "ci_win_rate": ci95(repeat_win_rates),
            "mean_loss_rate": float(np.mean(repeat_loss_rates)),
            "mean_draw_rate": float(np.mean(repeat_draw_rates)),
            "mean_time": float(np.mean(repeat_times)),
            "ci_time": ci95(repeat_times),
        })

df_resource_group = pd.DataFrame(resource_rows)

df_resource_group


In [ ]:
df_resource_robust = (
    df_resource_group
    .groupby(["config", "num_iterations"])
    .agg(
        avg_win_rate=("mean_win_rate", "mean"),
        worst_win_rate=("mean_win_rate", "min"),
        avg_loss_rate=("mean_loss_rate", "mean"),
        avg_time=("mean_time", "mean"),
        max_time=("mean_time", "max"),
    )
    .reset_index()
)

df_resource_robust["robust_score"] = (
    df_resource_robust["worst_win_rate"]
    + 0.50 * df_resource_robust["avg_win_rate"]
    - 0.05 * df_resource_robust["avg_time"]
)

df_resource_robust = df_resource_robust.sort_values(
    ["robust_score", "worst_win_rate", "avg_win_rate", "avg_time"],
    ascending=[False, False, False, True],
)

df_resource_robust


In [ ]:
plot_df = df_resource_robust.sort_values("num_iterations")

plt.figure(figsize=(9, 5))

plt.plot(
    plot_df["num_iterations"],
    plot_df["avg_win_rate"],
    marker="o",
    linewidth=2,
    label="Win rate promedio",
)

plt.plot(
    plot_df["num_iterations"],
    plot_df["worst_win_rate"],
    marker="s",
    linestyle="--",
    linewidth=2,
    label="Peor caso",
)

plt.xlabel("Número de iteraciones MCTS")
plt.ylabel("Win rate")
plt.ylim(0, 1.05)
plt.title("Recursos vs desempeño robusto del agente Group A")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
plot_df = df_resource_robust.sort_values("num_iterations")

plt.figure(figsize=(9, 5))

plt.plot(
    plot_df["num_iterations"],
    plot_df["avg_time"],
    marker="o",
    linewidth=2,
    color=PRIMARY_COLOR,
)

plt.xlabel("Número de iteraciones MCTS")
plt.ylabel("Tiempo promedio por jugada (s)")
plt.title("Costo computacional asociado al número de iteraciones")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
resource_matrix = df_resource_group.pivot_table(
    index="num_iterations",
    columns="opponent",
    values="mean_win_rate",
    aggfunc="mean",
)

plt.figure(figsize=(10, 6))

plt.imshow(
    resource_matrix,
    aspect="auto",
    origin="lower",
)

plt.colorbar(label="Win rate promedio")

plt.xticks(
    ticks=np.arange(len(resource_matrix.columns)),
    labels=resource_matrix.columns,
    rotation=35,
    ha="right",
)

plt.yticks(
    ticks=np.arange(len(resource_matrix.index)),
    labels=resource_matrix.index,
)

plt.xlabel("Oponente")
plt.ylabel("Número de iteraciones")
plt.title("Desempeño por recursos y tipo de oponente")
plt.show()

resource_matrix


## 8. Conclusión automática


In [ ]:
best_agent = df_round_robin_summary.iloc[0]
best_resource = df_resource_robust.iloc[0]

random_row = df_round_robin[
    (df_round_robin["agent"] == "Group A policy")
    & (df_round_robin["opponent"] == "Random")
]

group_b_row = df_round_robin[
    (df_round_robin["agent"] == "Group A policy")
    & (df_round_robin["opponent"] == "Group B policy")
]

print("=== Conclusión grupal ===")
print()

print(
    f"El agente con mayor win rate promedio en la comparación round-robin fue "
    f"{best_agent['agent']}, con un win rate promedio de "
    f"{best_agent['mean_win_rate']:.2%} y un peor caso de "
    f"{best_agent['worst_win_rate']:.2%}."
)

print()

print(
    f"Para el agente Group A policy, la configuración robusta más favorable fue "
    f"{best_resource['config']} con {int(best_resource['num_iterations'])} iteraciones, "
    f"win rate promedio de {best_resource['avg_win_rate']:.2%}, "
    f"peor caso de {best_resource['worst_win_rate']:.2%} y tiempo promedio de "
    f"{best_resource['avg_time']:.4f} segundos por jugada."
)

print()

if not random_row.empty:
    print(
        f"Contra el agente aleatorio, Group A policy obtuvo un win rate de "
        f"{float(random_row['win_rate'].iloc[0]):.2%}, lo que confirma que supera "
        f"una línea base no estratégica."
    )

if not group_b_row.empty:
    print(
        f"Contra Group B policy, Group A policy obtuvo un win rate de "
        f"{float(group_b_row['win_rate'].iloc[0]):.2%}, lo que permite comparar "
        f"el desempeño relativo frente a otro agente del torneo."
    )

print()

print(
    "La comparación no solo permite afirmar la superioridad relativa del agente escogido, "
    "sino también caracterizar el potencial del grupo: los agentes con mejor desempeño "
    "mantienen buen win rate promedio y mejor peor-caso ante múltiples oponentes, mientras "
    "que las configuraciones con mayor número de iteraciones muestran el trade-off entre "
    "calidad de búsqueda y costo computacional."
)


## Texto técnico para el informe

Se realizó una comparación round-robin entre los agentes disponibles en la carpeta `groups`, incluyendo `Group A policy`, `Group B policy`, `Group C policy` cuando estuvo disponible, un agente aleatorio y un agente con preferencia por centro. Además, se evaluaron configuraciones del agente `Group A` variando una variable numérica de recursos: el número de iteraciones MCTS. Para cada combinación se alternaron colores y se midieron win rate promedio, peor caso, tasa de derrota, movimientos promedio y tiempo promedio por jugada.

La selección del agente no se basó únicamente en vencer al aleatorio, sino en su desempeño relativo contra una suite de oponentes. El criterio de robustez priorizó el peor win rate observado, luego el win rate promedio y finalmente el costo computacional. Esto permite concluir no solo la superioridad del agente escogido, sino también el potencial comparativo de los agentes del grupo bajo distintos presupuestos de búsqueda y diferentes estilos de oponente.
